# 04_gov_<agreement>_<dataset>_<table>

Governance enrichment notebook that uses profiling evidence from 02/03 and human review before metadata writes.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit import (
    draft_business_context,
    draft_governance,
    extract_column_business_context_suggestions,
    extract_governance_suggestions,
    get_reviewed_business_context_rows,
    prepare_business_context_profile_input,
    prepare_governance_input,
    read_lakehouse_table,
    register_current_notebook,
    review_business_context,
    review_governance,
    write_business_context,
    write_governance,
)


In [ ]:
agreement_id = "agreement_sales_orders"
env_name = ENV
dataset_name = "sales"
table_name = "orders"
save_to_metadata = True
register_notebook_to_metadata = True


In [ ]:
agreements_df = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_DATA_AGREEMENT")
agreement_ids = {str(r.get("agreement_id") or r.get("AGREEMENT_ID") or "") for r in agreements_df.toPandas().to_dict("records")}
if agreement_id not in agreement_ids:
    raise ValueError(f"agreement_id not found in METADATA_DATA_AGREEMENT: {agreement_id}")


In [ ]:
if register_notebook_to_metadata:
    register_current_notebook(
        spark=spark,
        metadata_path=CONFIG.paths[env_name]["metadata"],
        agreement_id=agreement_id,
        notebook_type="04_gov",
        environment_name=env_name,
        dataset_name=dataset_name,
        table_name=table_name,
    )


In [ ]:
profile_df = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_PROFILE_ROWS")
profile_rows = profile_df.toPandas().to_dict("records")
filtered_profile_rows = [
    row for row in profile_rows
    if str(row.get("environment_name") or row.get("ENVIRONMENT_NAME") or "") == env_name
    and str(row.get("dataset_name") or row.get("DATASET_NAME") or "") == dataset_name
    and str(row.get("table_name") or row.get("TABLE_NAME") or "") == table_name
    and (
        "agreement_id" not in row
        or str(row.get("agreement_id") or "") == agreement_id
        or str(row.get("AGREEMENT_ID") or "") == agreement_id
    )
]


In [ ]:
prepared_context_rows = prepare_business_context_profile_input(
    profile_rows=filtered_profile_rows,
    table_name=table_name,
    table_context=f"{dataset_name}.{table_name}",
)
prepared_context_df = spark.createDataFrame(prepared_context_rows)
context_ai_df = draft_business_context(
    prepared_context_df,
    prompt_template=AI_PROMPTS.business_context_prompt_template,
)
context_suggestions = extract_column_business_context_suggestions(context_ai_df)


In [ ]:
review_business_context(
    suggestions=context_suggestions,
    environment_name=env_name,
    dataset_name=dataset_name,
    table_name=table_name,
)


In [ ]:
approved_context_rows = get_reviewed_business_context_rows("approved")
if save_to_metadata and approved_context_rows:
    write_business_context(
        spark=spark,
        rows=approved_context_rows,
        metadata_path=CONFIG.paths[env_name]["metadata"],
        table_name="METADATA_COLUMN_CONTEXT",
    )


In [ ]:
prepared_governance_rows = prepare_governance_input(
    profile_rows=filtered_profile_rows,
    table_name=table_name,
    column_contexts=approved_context_rows,
)
prepared_governance_df = spark.createDataFrame(prepared_governance_rows)
governance_ai_df = draft_governance(
    prepared_governance_df,
    prompt=AI_PROMPTS.governance_review_prompt_template,
)
governance_suggestions = extract_governance_suggestions(governance_ai_df)


In [ ]:
review_governance(
    suggestions=governance_suggestions,
    environment_name=env_name,
    dataset_name=dataset_name,
    table_name=table_name,
)


In [ ]:
if save_to_metadata:
    write_governance(
        spark=spark,
        metadata_path=CONFIG.paths[env_name]["metadata"],
        approved_rows=None,
        table_name="METADATA_COLUMN_GOVERNANCE",
    )


In [ ]:
# Optional read-only visibility for DQ rules
metadata_dq_rules = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_DQ_RULES")
display(metadata_dq_rules)
